[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jibby2k1/SPS_Curriculum/blob/main/Intro_DSP/Sigma_Delta_Quantization.ipynb)


**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Sigma-Delta & Quantization Theory

How a **1-bit** converter delivers 16+ bits of audio: oversampling spreads quantization noise thin, and the ΣΔ loop *shapes* it out of the band you care about. The bridge between [Real-Time DSP's](./Real_Time_DSP.ipynb) fixed-point world and actual converter hardware — with the 6 dB/bit and noise-shaping laws measured, not recited.

## 1. Pre-requisites

[Real-Time DSP](./Real_Time_DSP.ipynb) S1 (Q-format), [FoSP2](./Foundations_of_Signal_Processing_2.ipynb) S2 (oversampling/decimation), [Statistical SP](./Statistical_Signal_Processing.ipynb) S2 (PSD).

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal as sig
rng = np.random.default_rng(0)

---
### 🕐 Session 1 of 2 — *Quantization Noise & the Oversampling Dividend* (~40 min)
**Goal:** the white-noise model of quantization, its 6 dB/bit law, and +3 dB per octave of oversampling.
**Builds on:** [Real-Time DSP](./Real_Time_DSP.ipynb) S1. &nbsp; **Feeds into:** Session 2 (noise shaping).

---

<details>
<summary>🎓 <b>Teacher notes — Session 1: Quantization Noise & the Oversampling Dividend</b></summary>

**Timing (~40 min).** 10 min the white-noise model and its assumptions · 8 min the 6 dB/bit law · 12 min the demo, including both discrepancies · 8 min why oversampling alone is not enough · 2 min buffer.

**Board first — the fixed budget.** Quantization error has total power $\Delta^2/12$, set by the *step size alone*. Not by the sampling rate, not by the signal. Write that on the board and leave it there, because the entire workshop is about the two things you can do with a fixed budget: spread it thinner (Session 1) or move it somewhere harmless (Session 2). Framing both sessions as operations on one conserved quantity is what makes ΣΔ feel inevitable rather than clever.

**State the white-noise model as an approximation with conditions.** "Quantization error behaves like white noise" holds when the signal is busy enough to traverse many steps between samples. It fails for slowly-varying or heavily-oversampled inputs, where consecutive errors become correlated and appear as *tones* rather than a noise floor — audible as idle patterns in real converters, and the reason dither exists. This is not a footnote: it explains the second discrepancy in the demo below, and it is why production designs deliberately add noise.

**The 6 dB/bit arithmetic worth doing aloud.** One extra bit halves the step, which quarters the noise power, which is 6.02 dB. Have the room derive it rather than memorise it. Then CD audio at 16 bits gives about 96 dB of dynamic range, and a 12-bit ADC about 72.

**Both rows of this demo deviate from the printed theory — use that, do not hide it.** The 6 dB/bit block measures ~0.9 dB *below* the printed rule in every row, because the rule assumes a full-scale sine and ours has amplitude 0.9. That is $20\log_{10}(0.9) = -0.92$ dB, and correcting the rule to $6.02b + 0.84$ matches the measurements to within 0.6 dB at 4 bits and 0.01 dB at 12. Point out the convergence too: the white-noise model gets *better* with more bits, which is exactly what the theory predicts.

If your students also did [Real-Time DSP](./Real_Time_DSP.ipynb), this is the same lesson with the opposite sign — there the measurement came out 2.93 dB *above* the rule. Both times the cause is the full-scale convention, not a broken system. A rule of thumb that is off by a constant is nearly always a convention mismatch; chase the constant before you chase the code.

**The oversampling row deviates too, and further than the amplitude explains.** Measured shifts are −6.9 dB at 4× and −17.5 dB at 16×, against −6.0 and −12.0 predicted. The 16× row *beats* its prediction by 5.5 dB, which is not accounted for above. The likely cause is the white-noise model breaking down under heavy oversampling — a 997 Hz sine sampled at 768 kHz moves less than one quantization step between samples, so the error becomes strongly correlated and concentrates into harmonics rather than spreading uniformly, and some of those land outside the audio band. Present this as an open thread rather than a settled explanation; it has not been verified here, and "check whether the error spectrum is actually flat" is a genuinely good exercise. Modelling assumptions are worth more scepticism than the numbers they predict.

**Close on why this session is not the answer.** Half a bit per octave is real but hopeless as a strategy: 16 bits from 1 bit would need $2^{30}$× oversampling. Say the number out loud — it makes Session 2's loop land as a necessity rather than an optimisation.
</details>

## 2. Noise You Can Dilute

💡 **Intuition.** Quantization error behaves like white noise of total power $\Delta^2/12$ — and that total is **fixed by the step size**, not by the sampling rate. Sample $M×$ faster and the same noise power spreads over $M×$ the bandwidth: the slice sitting on your signal band shrinks by $M$ — **+3 dB of SNR per octave of oversampling** (half a bit per octave), redeemed by a [decimating low-pass filter](./Foundations_of_Signal_Processing_2.ipynb). Useful, but slow: 16 bits from 1 bit would need $2^{30}$× oversampling. Session 2 does radically better.

In [2]:
# the two laws, measured
def quantize(x, bits):
    L = 2**(bits-1)
    return np.clip(np.round(x*L), -L, L-1)/L

fs_base = 48_000
t = np.arange(2**16)/fs_base
x = 0.9*np.sin(2*np.pi*997*t)                        # 997 Hz: coprime-ish with fs, avoids spectral pileup

print("6 dB/bit (measured in the full band):")
for b in [4, 8, 12]:
    e = quantize(x, b) - x
    print(f"  {b:2d} bits: SNR {10*np.log10(np.var(x)/np.var(e)):5.1f} dB   (theory ≈ {6.02*b+1.76:.1f})")

print("\n+3 dB per octave of oversampling (8-bit quantizer, 24 kHz signal band):")
for M in [1, 4, 16]:
    fs_os = fs_base*M
    t_os = np.arange(2**16)/fs_os
    x_os = 0.9*np.sin(2*np.pi*997*t_os)
    q = quantize(x_os, 8)
    # measure noise power INSIDE the audio band only
    f, P = sig.welch(q - x_os, fs=fs_os, nperseg=4096)
    inband = P[f < fs_base/2].sum() * (f[1]-f[0])
    print(f"  {M:2d}x oversampled: in-band quantization noise {10*np.log10(inband):6.1f} dB   "
          f"({0 if M==1 else -10*np.log10(M):+.1f} dB expected shift)")

6 dB/bit (measured in the full band):
   4 bits: SNR  25.5 dB   (theory ≈ 25.8)
   8 bits: SNR  49.1 dB   (theory ≈ 49.9)
  12 bits: SNR  73.1 dB   (theory ≈ 74.0)

+3 dB per octave of oversampling (8-bit quantizer, 24 kHz signal band):
   1x oversampled: in-band quantization noise  -53.1 dB   (+0.0 dB expected shift)
   4x oversampled: in-band quantization noise  -60.0 dB   (-6.0 dB expected shift)
  16x oversampled: in-band quantization noise  -70.6 dB   (-12.0 dB expected shift)


**What just happened.** Two laws measured — and both deviate from the printed theory, which is the more instructive outcome.

**The 6 dB/bit law, and its 0.9 dB offset.** Measured 25.5 / 49.1 / 73.1 dB against a printed rule predicting 25.8 / 49.9 / 74.0. Every row falls about 0.9 dB short, and the constant offset is the giveaway: $6.02b + 1.76$ assumes a sine that *fills full scale*, and ours has amplitude 0.9. That costs exactly $20\log_{10}(0.9) = -0.92$ dB. Correct the rule to $6.02b + 0.84$ and the predictions become 24.9 / 49.0 / 73.1 — matching the measurements to 0.6 dB at 4 bits, 0.1 dB at 8, and 0.01 dB at 12.

The shrinking residual is itself informative. The $\Delta^2/12$ white-noise model assumes the error is uniformly distributed across a step and uncorrelated between samples, which becomes truer as steps get finer. At 4 bits there are only 16 levels and the model is visibly approximate; by 12 bits it is exact to a hundredth of a decibel. **The step-per-bit law is solid; the additive constant depends entirely on conventions you must check.** If you also did [Real-Time DSP](./Real_Time_DSP.ipynb), note this is the same lesson with the sign reversed — there the measurement came out 2.93 dB *above* the rule, for the same category of reason.

**The oversampling law, which over-performs.** Measured in-band noise: −53.1 dB at 1×, −60.0 at 4×, −70.6 at 16×. That is a shift of −6.9 dB and −17.5 dB, against −6.0 and −12.0 predicted. The 4× row is close; the 16× row beats its prediction by **5.5 dB**, and the amplitude correction above does not explain it.

Worth being clear that this is not fully accounted for here. The most likely cause is the white-noise model failing under heavy oversampling: at 16× the sine is sampled at 768 kHz, so it moves less than one quantization step between consecutive samples, the error becomes strongly correlated rather than white, and it concentrates into harmonics of 997 Hz instead of spreading uniformly — with some of that energy landing above the audio band and escaping the in-band sum. That is a plausible mechanism, not a verified one. Plotting the error spectrum at 1× and 16× would settle it, and doing so is a better exercise than accepting the explanation.

**What survives regardless.** Oversampling genuinely does dilute in-band quantization noise, and the direction and rough magnitude hold. But +3 dB per octave is a hopeless route to high resolution on its own: reaching 16 bits from 1 bit would need about $2^{30}$× oversampling. The noise budget is fixed and spreading it thinner has sharply diminishing returns. Session 2 stops spreading the noise and starts *moving* it.

---
### 🕐 Session 2 of 2 — *Noise Shaping: the ΣΔ Loop* (~40 min)
**Goal:** put the quantizer in a feedback loop: same noise total, pushed out of band — 1 bit becomes hi-fi.
**Builds on:** Session 1.

---

<details>
<summary>🎓 <b>Teacher notes — Session 2: Noise Shaping — the ΣΔ Loop</b></summary>

**Timing (~40 min).** 10 min the loop and its transfer functions · 10 min the shaped spectrum · 12 min decimation and the effective-bits table · 8 min what production designs add.

**Board first — solve the loop, it is four lines.** Write the modulator with the quantizer replaced by "signal plus additive error $e$." Push it through and you get output = signal + $(1 - z^{-1})e$ for first order. Two things fall out at once: the **signal transfer function is 1** (the signal passes untouched) and the **noise transfer function is a high-pass** (a zero at $z = 1$, i.e. at DC). Students who see those two separately never confuse noise shaping with filtering the signal.

**The one-sentence intuition.** $(1 - z^{-1})$ is a differencer, so near DC the loop is subtracting its own previous mistake — the integrator remembers the error it just made and corrects for it next sample. High frequencies get no such cancellation, which is where the noise goes. And crucially: *total* noise power is unchanged or slightly increased. Nothing is removed. It is relocated to a band you were going to decimate away anyway, which is why oversampling and shaping are a package rather than two independent tricks.

**Ask the room before running the decimation cell.** "Plain 1-bit at 64× oversampling — how many effective bits?" People guess 3 or 4. It is **−5.8 dB SNR**, i.e. *negative* effective bits: the output has more error than signal. That result is the best argument in the workshop, because it isolates the loop as the active ingredient. Same 1 bit, same 64× oversampling, same decimator — only the feedback differs, and it is worth 75 dB.

**The order arithmetic.** First order buys 9 dB/octave of oversampling, second order 15. Measured here: 46.7 dB (7.5 bits) and 69.5 dB (11.3 bits). Ask why not just keep raising the order — the answer is stability. Loops above second order are conditionally stable and can enter large-amplitude oscillation; real designs use careful coefficient scaling, or cascaded (MASH) structures that stack stable low-order stages instead. This is a genuine engineering constraint, not a detail.

**Point at the loop code honestly.** `sigma_delta_1bit` uses `v[k-1]` with a `k == 0` guard — a small, readable implementation, and note that at $k=0$ Python's negative indexing would otherwise wrap to the last element, which is why the guard exists. Also flag that the second-order branch is a simple cascade of two integrators without the coefficient scaling a production modulator would carry; it is correct for this demonstration and would need scaling to stay stable across all inputs.

**Close on the reframe, which is the point of the whole workshop.** A hard *analog* problem — build a precise multi-level converter, with matched resistors and tight tolerances — was replaced by a fast 1-bit comparator plus a *digital* filter. That is why converter datasheets read like DSP homework, and why the [FPGA workshop's](../Intro_FPGA/Intro_FPGA.ipynb) CIC decimators exist. Trading analog precision for digital signal processing is one of the highest-leverage moves in the field.

**Say what production adds.** Higher order, dither to break up idle tones (the correlated-error problem from Session 1), and multi-bit internal quantizers. 11.3 effective bits here versus 16–24 in a real part is a fair gap to acknowledge — the demo shows the mechanism, not the state of the art.
</details>

## 3. The Loop That Cheats

💡 **Intuition.** Wrap the quantizer in feedback: integrate the error before quantizing, subtract the output. Solve the loop and the signal passes untouched while the quantization noise is multiplied by $(1 - z^{-1})$ — a **high-pass**: near DC the loop's memory cancels its own past mistakes. Total noise unchanged; its *location* moved to high frequencies you were going to [decimate away anyway](./Foundations_of_Signal_Processing_2.ipynb). First-order shaping buys 9 dB/octave; second-order, 15 — which is how a 1-bit stream at 64× oversampling delivers CD-quality audio (DSD, and virtually every audio ADC/DAC you own).

In [3]:
def sigma_delta_1bit(x, order=1):
    """1-bit ΣΔ modulator, first or second order."""
    v = np.zeros(len(x)); i1 = i2 = 0.0
    for k in range(len(x)):
        if order == 1:
            i1 += x[k] - v[k-1] if k else x[k]
            v[k] = 1.0 if i1 >= 0 else -1.0
        else:
            e = x[k] - v[k-1] if k else x[k]
            i1 += e
            i2 += i1 - v[k-1] if k else i1
            v[k] = 1.0 if i2 >= 0 else -1.0
    return v

M = 64
fs_os = fs_base*M
t_os = np.arange(2**18)/fs_os
x_os = 0.5*np.sin(2*np.pi*997*t_os)

plain_1bit = np.sign(x_os)                                # 1-bit quantizer, no loop
sd1 = sigma_delta_1bit(x_os, 1)
sd2 = sigma_delta_1bit(x_os, 2)

plt.figure(figsize=(9, 3))
for y, name in [(plain_1bit, "plain 1-bit"), (sd1, "ΣΔ 1st order"), (sd2, "ΣΔ 2nd order")]:
    f, P = sig.welch(y - x_os, fs=fs_os, nperseg=8192)
    plt.semilogx(f, 10*np.log10(P + 1e-16), label=name, linewidth=0.9)
plt.axvline(fs_base/2, color="k", linestyle=":", linewidth=1)
plt.text(fs_base/2, -60, " audio band edge", fontsize=7)
plt.legend(fontsize=8); plt.xlabel("Hz"); plt.ylabel("error PSD [dB/Hz]")
plt.title("noise SHAPING: same total error, swept out of the audio band")
plt.tight_layout(); plt.show()

/tmp/ipykernel_2993110/1785049913.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


**What just happened.** Three error spectra from the same 1-bit quantizer. The plain quantizer's error is roughly flat — noise spread evenly, including right across the audio band. The first-order ΣΔ curve tilts upward with frequency, sitting well below the flat line at low frequencies and above it at high ones. The second-order curve tilts *harder*: even lower in the audio band, even higher outside it.

**Nothing was removed — read the curves as a redistribution.** Where the shaped traces dip below the flat one at low frequency, they rise above it at high frequency, and the total area is not reduced (second-order shaping slightly *increases* total error power). That is the point: the noise budget from Session 1 is conserved, and the loop only chooses where to spend it. Solving the loop shows why — the output is signal $+\,(1 - z^{-1})e$ for first order, so the signal transfer function is 1 while the noise gets multiplied by a high-pass with a zero at DC. The integrator remembers the error it just made and cancels it next sample, which works near DC and not at all near Nyquist.

**The dotted line is the whole argument.** Everything to its right gets thrown away by the decimator, so noise pushed there costs nothing. Oversampling and noise shaping are therefore not two independent tricks — shaping is only useful *because* there is out-of-band room to dump into, and that room is what 64× oversampling bought. Neither works alone: Session 1 showed oversampling alone needs $2^{30}$×, and shaping without oversampling would have nowhere to put the noise.

Note the steepness ordering — second order tilts more than first, and that slope is the effective-bits number. First-order shaping is worth 9 dB per octave of oversampling, second-order 15, against oversampling's bare 3. That is the difference between hopeless and shipping.

The next cell stops reading slopes off a plot and does the honest thing: run each stream through a real decimator and measure the audio-band SNR that actually results.

In [4]:
# redeem the promise: decimate each 1-bit stream back to 48 kHz and measure audio-band SNR
def audio_snr(y):
    dec = sig.decimate(sig.decimate(y, 8, ftype="fir"), 8, ftype="fir")   # 64x → 1x
    ref = sig.decimate(sig.decimate(x_os, 8, ftype="fir"), 8, ftype="fir")
    return 10*np.log10(np.var(ref)/np.var(dec - ref))

for y, name in [(plain_1bit, "plain 1-bit @64x"), (sd1, "ΣΔ 1st order"), (sd2, "ΣΔ 2nd order")]:
    snr = audio_snr(y)
    print(f"{name:18s}: audio-band SNR after decimation {snr:5.1f} dB  (~{(snr-1.76)/6.02:.1f} effective bits)")
print("\n→ ONE physical bit, second-order shaping, 64x oversampling ≈ a 12+ bit converter;")
print("  production designs add order, dither, and multibit stages to reach 16–24 bits")

plain 1-bit @64x  : audio-band SNR after decimation  -5.8 dB  (~-1.3 effective bits)
ΣΔ 1st order      : audio-band SNR after decimation  46.7 dB  (~7.5 effective bits)
ΣΔ 2nd order      : audio-band SNR after decimation  69.5 dB  (~11.3 effective bits)

→ ONE physical bit, second-order shaping, 64x oversampling ≈ a 12+ bit converter;
  production designs add order, dither, and multibit stages to reach 16–24 bits


**What just happened.** The promise, redeemed and measured after actual decimation back to 48 kHz:

| | audio-band SNR | effective bits |
|---|---|---|
| plain 1-bit @ 64× | **−5.8 dB** | −1.3 |
| ΣΔ 1st order | **46.7 dB** | 7.5 |
| ΣΔ 2nd order | **69.5 dB** | 11.3 |

**Start with the negative number, because it is the strongest result here.** Plain 1-bit at 64× oversampling gives −5.8 dB — *more error power than signal power*, worse than useless. Oversampling by 64 on its own bought essentially nothing, exactly as Session 1's +3 dB/octave arithmetic predicted it would. This row is what makes the experiment controlled: same single bit, same 64× rate, same decimator, and the only difference in the rows below is the feedback loop. That loop is worth **75 dB**.

**One physical bit, eleven effective ones.** The comparator still answers a single yes/no question per sample. The extra resolution is not in the hardware — it is manufactured by feedback and filtering, then collected by the decimator. That is the conceptual payoff of the workshop: a hard *analog* problem (build a precise multi-level converter with matched components) was traded for a fast 1-bit comparator plus a *digital* filter, and digital filters are cheap and exact while analog precision is neither. This is why converter datasheets read like DSP homework, and why the [FPGA workshop's](../Intro_FPGA/Intro_FPGA.ipynb) CIC decimators exist.

**Why not simply keep raising the order?** Stability. Modulators above second order are only conditionally stable and can fall into large-amplitude oscillation from which the loop does not recover. Production designs handle this with careful coefficient scaling or with MASH structures that cascade stable low-order stages rather than building one high-order loop. The 9 → 15 dB/octave progression is real but it does not extrapolate freely.

**And the honest gap.** 11.3 effective bits is not the 16–24 a real audio converter delivers. Closing that gap takes higher-order loops, multi-bit internal quantizers, and **dither** — deliberately added noise that breaks up the correlated-error idle tones flagged in Session 1, which are audible as whistles on quiet passages and are exactly the failure of the white-noise model that made that session's 16× row misbehave. The demo shows the mechanism at work, not the state of the art, and the second-order loop here carries none of the coefficient scaling a shipping design would need.

## 4. Conclusion

Quantization noise has a fixed budget; oversampling dilutes it (+3 dB/octave, measured), and the ΣΔ loop *relocates* it (effective bits measured climbing with loop order). The dirty analog problem became a [multirate filtering](./Foundations_of_Signal_Processing_2.ipynb) problem — which is why converter datasheets read like DSP homework.

---
## Where next

- [Real-Time DSP](./Real_Time_DSP.ipynb) — where the decimated samples land.
- [Intro to FPGA](../Intro_FPGA/Intro_FPGA.ipynb) — CIC decimators: the hardware that does this at GHz.
- [Model Compression](../Intro_Mach_Learn/Model_Compression.ipynb) — the same quantization mathematics, aimed at neural weights.